In [1]:
import pandas as pd
from pathlib import Path
from datetime import datetime, timedelta
# import math
import plotly.express as px
# import plotly.graph_objects as go

remember:
- check for duplicates in sub-dfs
- function for creating sub-dfs
- think about placement for the functions - revise order
- merge plotting with creating the sub-df
- collect data from year prior to 08/12/24
- acled login credentials - update and share

In [2]:
dir_processed = Path('../data/processed')
dir_processed.mkdir(exist_ok=True)
dir_raw = Path('../data/raw')
dir_raw.mkdir(exist_ok=True)

In [11]:
def df_index(dataframe):
    rows = len(dataframe)

    dataframe.index = range(1, rows+1)
    
    return dataframe

In [22]:
def color_map_grey(category):
    colors = [
        "#C5CDD5", 
        "#C4CCD0", 
        "#D0CBC2",
        "#CBCBCB", 
        "#B8B8B8",
        "#C8C1B6", 
        "#C1C1C1", 
        "#A9A9A9", 
        "#B6AE9E",
        "#B0BAC5", 
        "#BBC3CD", 
        "#ACACAC", 
        "#909EAE", 
        "#B5BEC9", 
        "#808080", 
        "#6082B6", 
        "#4F4F4F",
        "#A49A87",
        "#87A0A4",
        "#C8CCD5"
        ] 
    return {event: colors[i] for i, event in enumerate(category)}

In [3]:
"refactored return statement after Lecture W03D04"

def color_map_pop(selected_category, category_list):
    return {event: "#5C8DC5" if category_list[i] == selected_category else "#D3D3D3" for i, event in enumerate(category_list)}

In [4]:
"refactored return statement after Lecture W03D04"

def color_map_pop_multiple(selected_categories, category_list):
        return {admin: "#5C8DC5" if admin in selected_categories else "#D3D3D3" for i, admin in enumerate(category_list)}

In [5]:
def event_names(df, column):
    return list(df[column].unique())

In [ ]:
dataset = pd.read_csv(dir_processed / 'dataset.csv')
dataset.index = dataset.index + 1
dataset.head(30)

Part 1: Analyse conflict event types since the regime fall

In [8]:
dataset_grouped = (
    dataset
    .groupby(['interval', 'event_type'], as_index=False)
    .agg(event_count=('event_type', 'size'))
)


In [9]:
dataset_dates = dataset[['interval', 'start date', 'end date']]

In [ ]:
dataset_dates = dataset_dates.drop_duplicates()
df_index(dataset_dates)

In [ ]:
# dataset_grouped
data_grouped_with_dates = pd.merge(dataset_grouped, dataset_dates, on='interval')
data_grouped_with_dates.head(30)

In [117]:
dataset_grouped.to_csv(dir_processed/'counts_per_event_type.csv', index=False)

In [ ]:
def event_names(df, column):
    return list(df[column].unique())

In [ ]:
# names = event_names(dataset, 'event_type')
# names

['Strategic developments',
 'Violence against civilians',
 'Explosions/Remote violence',
 'Protests',
 'Battles',
 'Riots']

In [17]:
events_list = list(dataset['event_type'].unique())

In [ ]:
# ended up not using chart

plot_events_sdpop = px.bar(
    dataset_grouped, 
    x='interval', 
    y='event_count', 
    color='event_type',
    color_discrete_map=color_map_pop('Strategic developments', events_list),
    title='Conflict events since the regime overthrow, strategic developments have gradually become more prominent',
    labels={'interval': 'Months elapsed since 08/12/2024', 'event_count': 'Number of events'}
)

plot_events_sdpop.update_layout(
    title_font_size=14, font_size=10, yaxis=dict(nticks=10), 
    legend=dict(title=(dict(text="Events")))
)
plot_events_sdpop.update_xaxes(dtick=1)

In [124]:
plot_events_sdpop.write_html("../docs/assets/bar_events_sd_pop.html")

In [31]:
bubble_events = px.scatter(
    data_grouped_with_dates,
    x="start date", 
    y="event_count",
    size="event_count", 
    color="event_type",
    hover_name="event_type", 
    size_max=20, 
    color_discrete_map=color_map_grey(events_list), 
    labels={'start date':"Time elapsed since 08/12/2024", 'event_count':"Number of events"})


bubble_events.update_layout(
    title = 'Conflict events since the fall of the Assad regime',
    legend_title_text = "Events",    
    xaxis_tickformat = '%d %b %y',
    xaxis=dict(nticks=8),
    title_font_size=14,
    font_size=10,
)


In [26]:
bubble_events.write_html('../docs/assets/bubble_events.html')

In [27]:
# moved to NB03b

df_sd = (
    dataset
    .query("event_type == 'Strategic developments'")
    .groupby(['interval', 'sub_event_type'], as_index=False)
    .agg(event_count=('sub_event_type', 'size'))
)

In [ ]:
# color_map_SD_pop = {
#     'Non-violent transfer of territory': '#6082B6',
#     "Agreement": "#D3D3D3",
#     "Headquarters or base established": "#D3D3D3",
#     "Change to group/activity": "#D3D3D3",
#     "Arrests": "#D3D3D3",
#     'Disrupted weapons use': "#D3D3D3",
#     "Looting/property destruction": "#D3D3D3",
#     "Other": "#D3D3D3"
# }

 
# color_map_SD = {
#     'Non-violent transfer of territory': "#899499",
#     "Agreement": "#D3D3D3",
#     "Headquarters or base established": '#708090',
#     "Change to group/activity": '#A9A9A9',
#     "Arrests": "#808080",
#     'Disrupted weapons use': '#6082B6',
#     "Looting/property destruction": "#4F4F4F",
#     "Other": "#D3D3D3"
# }

In [ ]:
sd_events_list = list(df_sd['sub_event_type'].unique())

In [ ]:
plot_sd_cga = px.bar(
    df_sd,
    x='interval',
    y='event_count',
    color='sub_event_type',
    color_discrete_map=color_map_grey(sd_events_list)
)
plot_sd_cga.show()
plot_sd_cga.update_xaxes(dtick=1, tickmode="linear")


In [44]:
# Convert the bar plots to bubble charts

bubble_sd = px.scatter(
    df_sd, 
    x="interval", 
    y="event_count",
    size="event_count", 
    color="sub_event_type",
    hover_name="sub_event_type", 
    size_max=20, 
    color_discrete_map=color_map_grey(sd_events_list),
    labels={'interval':"Months elapsed since 08/12/2024", 'event_count':"Number of events"}
    )
bubble_sd.update_xaxes(dtick=1, tickmode="linear")


In [ ]:
bubble_sd.write_html("../docs/assets/bubble_strat_dev.html")

Explosions and remote violence 

In [45]:
df_erv = (
    dataset
    .query("event_type == 'Explosions/Remote violence'")
    .groupby(['interval', 'sub_event_type'], as_index=False)
    .agg(event_count=('sub_event_type', 'size'))
)

In [ ]:
df_erv = pd.merge(df_erv, dataset_dates, on='interval')
df_erv

In [48]:
erv_events_list = list(df_erv['sub_event_type'].unique())

In [ ]:
df_erv = (
    dataset
    .query("event_type == 'Explosions/Remote violence'")
    .groupby(['interval', 'sub_event_type'], as_index=False)
    .agg(event_count=('sub_event_type', 'size'))
)

df_erv = pd.merge(df_erv, dataset_dates, on='interval')

df_erv

erv_events_list = list(df_erv['sub_event_type'].unique())

bar_erv = px.bar(
    df_erv,
    x='interval',
    y='event_count',
    color='sub_event_type',
    color_discrete_map=color_map_grey(erv_events_list)
)

bar_erv.update_xaxes(dtick=1)
